In [0]:
%pip install snowflake-connector-python pandas scikit-learn -q

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%restart_python

In [0]:
# ============================================================
# CELL 2: Importera bibliotek och anslut till Snowflake
# ============================================================

# snowflake.connector används för att prata med Snowflake från Python
import snowflake.connector

# pandas används för att hantera data som en tabell (DataFrame)
import pandas as pd

# getpass döljer lösenordet när du skriver det — syns inte i koden
from getpass import getpass

# ============================================================
# Anslutningsinställningar till Snowflake
# account  = din unika Snowflake-URL (utan .snowflakecomputing.com)
# user     = ditt användarnamn i Snowflake
# database = databasen vi skapade
# schema   = schemat där dbt lade FCT_FEATURES
# warehouse= beräkningsmotorn i Snowflake
# ============================================================
conn = snowflake.connector.connect(
    account='BHTUAZA-AH74419',
    user='GH0U1',          
    password=getpass('Snowflake-lösenord: '),
    database='NSL_KDD_DB',
    schema='DBT_JDOE',       # schemat dbt skapade
    warehouse='COMPUTE_WH'
)

print("Anslutning till Snowflake lyckades!")

Snowflake-lösenord:  [REDACTED]

Anslutning till Snowflake lyckades!


In [0]:
# ============================================================
# CELL 3: Hämta data från Snowflake
# ============================================================

# SQL-frågan hämtar all data från FCT_FEATURES
# Vi exkluderar session_id och label eftersom de inte är
# numeriska features — ML-modellen behöver bara siffror
query = """
    SELECT * FROM NSL_KDD_DB.DBT_JDOE.FCT_FEATURES
"""

# Läs in resultatet som en pandas DataFrame
# En DataFrame är som en Excel-tabell i Python
df = pd.read_sql(query, conn)

# Stäng anslutningen till Snowflake — frigör resurser
conn.close()

# Visa grundläggande information om datasetet
print(f"Antal rader: {len(df)}")
print(f"Antal kolumner: {len(df.columns)}")
print(f"\nKolumnnamn:\n{list(df.columns)}")
print(f"\nFörsta 3 raderna:")
df.head(3)

/home/spark-4cfc13ae-255d-4d67-964c-cb/.ipykernel/16001/command-5248184784049619-3955758301:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


Antal rader: 125973
Antal kolumner: 45

Kolumnnamn:
['SESSION_ID', 'LABEL', 'ATTACK_CATEGORY', 'IS_ATTACK', 'DURATION', 'PROTOCOL_TYPE', 'SERVICE', 'FLAG', 'SRC_BYTES', 'DST_BYTES', 'LAND', 'WRONG_FRAGMENT', 'URGENT', 'HOT', 'NUM_FAILED_LOGINS', 'LOGGED_IN', 'NUM_COMPROMISED', 'ROOT_SHELL', 'SU_ATTEMPTED', 'NUM_ROOT', 'NUM_FILE_CREATIONS', 'NUM_SHELLS', 'NUM_ACCESS_FILES', 'NUM_OUTBOUND_CMDS', 'IS_HOST_LOGIN', 'IS_GUEST_LOGIN', 'COUNT', 'SRV_COUNT', 'SERROR_RATE', 'SRV_SERROR_RATE', 'RERROR_RATE', 'SRV_RERROR_RATE', 'SAME_SRV_RATE', 'DIFF_SRV_RATE', 'SRV_DIFF_HOST_RATE', 'DST_HOST_COUNT', 'DST_HOST_SRV_COUNT', 'DST_HOST_SAME_SRV_RATE', 'DST_HOST_DIFF_SRV_RATE', 'DST_HOST_SAME_SRC_PORT_RATE', 'DST_HOST_SRV_DIFF_HOST_RATE', 'DST_HOST_SERROR_RATE', 'DST_HOST_SRV_SERROR_RATE', 'DST_HOST_RERROR_RATE', 'DST_HOST_SRV_RERROR_RATE']

Första 3 raderna:


,SESSION_ID,LABEL,ATTACK_CATEGORY,IS_ATTACK,DURATION,PROTOCOL_TYPE,SERVICE,FLAG,SRC_BYTES,DST_BYTES,LAND,WRONG_FRAGMENT,URGENT,HOT,NUM_FAILED_LOGINS,LOGGED_IN,NUM_COMPROMISED,ROOT_SHELL,SU_ATTEMPTED,NUM_ROOT,NUM_FILE_CREATIONS,NUM_SHELLS,NUM_ACCESS_FILES,NUM_OUTBOUND_CMDS,IS_HOST_LOGIN,IS_GUEST_LOGIN,COUNT,SRV_COUNT,SERROR_RATE,SRV_SERROR_RATE,RERROR_RATE,SRV_RERROR_RATE,SAME_SRV_RATE,DIFF_SRV_RATE,SRV_DIFF_HOST_RATE,DST_HOST_COUNT,DST_HOST_SRV_COUNT,DST_HOST_SAME_SRV_RATE,DST_HOST_DIFF_SRV_RATE,DST_HOST_SAME_SRC_PORT_RATE,DST_HOST_SRV_DIFF_HOST_RATE,DST_HOST_SERROR_RATE,DST_HOST_SRV_SERROR_RATE,DST_HOST_RERROR_RATE,DST_HOST_SRV_RERROR_RATE
0,tcp_ftp_data_SF,normal,normal,0,0,tcp,ftp_data,SF,491,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2,2,0.0,0.0,0.0,0.0,1.00,0.00,0.0,150,25,0.17,0.03,0.17,0.0,0.0,0.0,0.05,0.0
1,udp_other_SF,normal,normal,0,0,udp,other,SF,146,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,13,1,0.0,0.0,0.0,0.0,0.08,0.15,0.0,255,1,0.00,0.60,0.88,0.0,0.0,0.0,0.00,0.0
2,tcp_private_S0,neptune,dos,1,0,tcp,private,S0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,123,6,1.0,1.0,0.0,0.0,0.05,0.07,0.0,255,26,0.10,0.05,0.00,0.0,1.0,1.0,0.00,0.0


In [0]:
# ============================================================
# CELL 4: Förbered data för ML-modellen
# ============================================================

# Importera verktyg för ML
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, f1_score, precision_score, recall_score
import numpy as np

# ============================================================
# Välj ut numeriska features
# ML-modellen kan bara hantera siffror — inte text
# Vi exkluderar: session_id (text), label (text),
# attack_category (text), protocol_type/service/flag (text)
# ============================================================
numeriska_kolumner = [
    'DURATION', 'SRC_BYTES', 'DST_BYTES', 'LAND', 'WRONG_FRAGMENT',
    'URGENT', 'HOT', 'NUM_FAILED_LOGINS', 'LOGGED_IN', 'NUM_COMPROMISED',
    'ROOT_SHELL', 'SU_ATTEMPTED', 'NUM_ROOT', 'NUM_FILE_CREATIONS',
    'NUM_SHELLS', 'NUM_ACCESS_FILES', 'NUM_OUTBOUND_CMDS',
    'IS_HOST_LOGIN', 'IS_GUEST_LOGIN', 'COUNT', 'SRV_COUNT',
    'SERROR_RATE', 'SRV_SERROR_RATE', 'RERROR_RATE', 'SRV_RERROR_RATE',
    'SAME_SRV_RATE', 'DIFF_SRV_RATE', 'SRV_DIFF_HOST_RATE',
    'DST_HOST_COUNT', 'DST_HOST_SRV_COUNT', 'DST_HOST_SAME_SRV_RATE',
    'DST_HOST_DIFF_SRV_RATE', 'DST_HOST_SAME_SRC_PORT_RATE',
    'DST_HOST_SRV_DIFF_HOST_RATE', 'DST_HOST_SERROR_RATE',
    'DST_HOST_SRV_SERROR_RATE', 'DST_HOST_RERROR_RATE',
    'DST_HOST_SRV_RERROR_RATE'
]

# X = features (det modellen lär sig av)
# y = målvariabel (det modellen ska förutsäga)
X = df[numeriska_kolumner]
y = df['IS_ATTACK']  # 0 = normal, 1 = attack

# ============================================================
# Dela upp data i träning (80%) och test (20%)
# Modellen tränas på 80% och utvärderas på de återstående 20%
# random_state=42 gör att uppdelningen blir samma varje gång
# ============================================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ============================================================
# Normalisera features med StandardScaler
# Skalar om alla värden så de ligger på samma skala
# Viktigt eftersom src_bytes kan vara miljoner medan
# root_shell bara är 0 eller 1
#
# Med skalning behandlas alla features lika från start, och modellen kan själv avgöra vilka som faktiskt är viktiga
# ============================================================
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f"Träningsdata:  {X_train_scaled.shape[0]} rader")
print(f"Testdata:      {X_test_scaled.shape[0]} rader")
print(f"\nFördelning i träningsdata:")
print(f"  Normal (0): {(y_train == 0).sum()}")
print(f"  Attack (1): {(y_train == 1).sum()}")

Träningsdata:  100778 rader
Testdata:      25195 rader

Fördelning i träningsdata:
  Normal (0): 53921
  Attack (1): 46857


In [0]:
# ============================================================
# CELL 5: Träna två ML-modeller och jämför dem
# ============================================================

# ------------------------------------------------------------
# MODELL 1: Isolation Forest (oövervakad)
# Lär sig INTE från labels — hittar bara "konstiga" datapunkter
# Fungerar som att isolera outliers i datan
# contamination = andelen vi förväntar oss är attacker (~46%)
# ------------------------------------------------------------
iso_forest = IsolationForest(contamination=0.46, random_state=42)
iso_forest.fit(X_train_scaled)

# Isolation Forest returnerar 1 (normal) och -1 (anomali)
# Vi konverterar till 0 (normal) och 1 (attack) för jämförelse
iso_pred_raw = iso_forest.predict(X_test_scaled)
iso_pred = (iso_pred_raw == -1).astype(int)

print("=" * 50)
print("MODELL 1: Isolation Forest (oövervakad)")
print("=" * 50)
print(classification_report(y_test, iso_pred,
      target_names=['Normal', 'Attack']))

# ------------------------------------------------------------
# MODELL 2: Random Forest (övervakad)
# Lär sig från labels — vet vad som är normal och attack
# Bygger 100 beslutsträd och röstar om svaret
# n_estimators = antal träd
# random_state = reproducerbart resultat
# ------------------------------------------------------------
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42
)
rf_model.fit(X_train_scaled, y_train)
rf_pred = rf_model.predict(X_test_scaled)

print("\n" + "=" * 50)
print("MODELL 2: Random Forest (övervakad)")
print("=" * 50)
print(classification_report(y_test, rf_pred,
      target_names=['Normal', 'Attack']))

# ------------------------------------------------------------
# Jämförelse av F1-score
# F1 är ett balanserat mått på precision och recall
# Precision = när modellen säger "attack", hur ofta har den rätt?
# Recall    = av alla verkliga attacker, hur många hittar modellen?
# ------------------------------------------------------------
iso_f1 = f1_score(y_test, iso_pred)
rf_f1  = f1_score(y_test, rf_pred)

print("\n" + "=" * 50)
print("JÄMFÖRELSE")
print("=" * 50)
print(f"Isolation Forest F1-score: {iso_f1:.3f}")
print(f"Random Forest    F1-score: {rf_f1:.3f}")
print(f"\nVinnare: {'Random Forest' if rf_f1 > iso_f1 else 'Isolation Forest'}")

MODELL 1: Isolation Forest (oövervakad)
              precision    recall  f1-score   support

      Normal       0.69      0.69      0.69     13422
      Attack       0.65      0.64      0.64     11773

    accuracy                           0.67     25195
   macro avg       0.67      0.67      0.67     25195
weighted avg       0.67      0.67      0.67     25195


MODELL 2: Random Forest (övervakad)
              precision    recall  f1-score   support

      Normal       0.99      1.00      1.00     13422
      Attack       1.00      0.99      1.00     11773

    accuracy                           1.00     25195
   macro avg       1.00      1.00      1.00     25195
weighted avg       1.00      1.00      1.00     25195


JÄMFÖRELSE
Isolation Forest F1-score: 0.642
Random Forest    F1-score: 0.995

Vinnare: Random Forest


In [0]:
# ============================================================
# CELL 6: Logga experiment med MLflow
# ============================================================

# mlflow är inbyggt i Databricks — inget behöver installeras
# Det fungerar som en dagbok för alla ML-experiment
import mlflow
import mlflow.sklearn

# Sätt vilket experiment vi loggar till
# Om det inte finns skapas det automatiskt
mlflow.set_experiment("/Users/tomthorsen1@gmail.com/nsl_kdd_intrusion_detection")

# ------------------------------------------------------------
# Logga Isolation Forest
# ------------------------------------------------------------
with mlflow.start_run(run_name="Isolation_Forest"):

    # log_param sparar inställningar vi använde
    mlflow.log_param("model_type", "IsolationForest")
    mlflow.log_param("contamination", 0.46)

    # log_metric sparar resultaten
    mlflow.log_metric("f1_score", iso_f1)
    mlflow.log_metric("precision", precision_score(y_test, iso_pred))
    mlflow.log_metric("recall", recall_score(y_test, iso_pred))

    # log_model sparar själva modellen
    mlflow.sklearn.log_model(iso_forest, "isolation_forest_model")
    print("Isolation Forest loggad till MLflow!")

# ------------------------------------------------------------
# Logga Random Forest
# ------------------------------------------------------------
with mlflow.start_run(run_name="Random_Forest_v1"):

    mlflow.log_param("model_type", "RandomForest")
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("max_depth", 10)

    mlflow.log_metric("f1_score", rf_f1)
    mlflow.log_metric("precision", precision_score(y_test, rf_pred))
    mlflow.log_metric("recall", recall_score(y_test, rf_pred))

    mlflow.sklearn.log_model(rf_model, "random_forest_model")

    # Spara run_id — behövs för att ladda modellen senare
    rf_run_id = mlflow.active_run().info.run_id
    print(f"Random Forest loggad till MLflow!")
    print(f"Run ID: {rf_run_id}")

2026/06/11 08:23:13 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://dbc-40a283be-55c0.cloud.databricks.com/ml/experiments/1918190157109294/models/m-168df339560846f6b1566c43e84438e7?o=7474659315390665
2026/06/11 08:23:18 INFO mlflow.models.model: Model logged without a signature. Signatures are required for Databricks UC model registry as they validate model inputs and denote the expected schema of model outputs. Please set `input_example` parameter when logging the model to auto infer the model signature. To manually set the signature, please visit https://www.mlflow.org/docs/3.8.1/ml/model/signatures.html for instructions on setting signature on models.


Isolation Forest loggad till MLflow!


2026/06/11 08:23:19 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://dbc-40a283be-55c0.cloud.databricks.com/ml/experiments/1918190157109294/models/m-884e074c6c374cbfa592f96c5c2a0a7a?o=7474659315390665
2026/06/11 08:23:22 INFO mlflow.models.model: Model logged without a signature. Signatures are required for Databricks UC model registry as they validate model inputs and denote the expected schema of model outputs. Please set `input_example` parameter when logging the model to auto infer the model signature. To manually set the signature, please visit https://www.mlflow.org/docs/3.8.1/ml/model/signatures.html for instructions on setting signature on models.


Random Forest loggad till MLflow!
Run ID: bb280d8c792940788072c972efe381e5


In [0]:
# ============================================================
# CELL 7: A/B-testning — jämför Random Forest v1 vs v2
# ============================================================
# A/B-testning innebär att vi kör två versioner parallellt
# och låter data avgöra vilken som är bäst
# Precis som Netflix testar två olika startsidor på olika användare

# ------------------------------------------------------------
# Version B: Random Forest med andra parametrar
# Vi ökar antalet träd (200 vs 100) och djupet (15 vs 10)
# för att se om det ger bättre resultat
# ------------------------------------------------------------
rf_model_v2 = RandomForestClassifier(
    n_estimators=200,   # fler träd än v1 (100)
    max_depth=15,       # djupare träd än v1 (10)
    random_state=42
)
rf_model_v2.fit(X_train_scaled, y_train)
rf_pred_v2 = rf_model_v2.predict(X_test_scaled)

# Beräkna metrics för v2
rf_f1_v2        = f1_score(y_test, rf_pred_v2)
rf_precision_v2 = precision_score(y_test, rf_pred_v2)
rf_recall_v2    = recall_score(y_test, rf_pred_v2)

# Logga v2 till MLflow
with mlflow.start_run(run_name="Random_Forest_v2"):
    mlflow.log_param("model_type", "RandomForest")
    mlflow.log_param("n_estimators", 200)
    mlflow.log_param("max_depth", 15)
    mlflow.log_metric("f1_score", rf_f1_v2)
    mlflow.log_metric("precision", rf_precision_v2)
    mlflow.log_metric("recall", rf_recall_v2)
    mlflow.sklearn.log_model(rf_model_v2, "random_forest_model")
    rf_run_id_v2 = mlflow.active_run().info.run_id

# ------------------------------------------------------------
# Simulera A/B-test med trafikdelning 70/30
# 70% av requests går till v1, 30% till v2
# Vi använder hash av radindex för att fördela trafiken
# deterministiskt (samma rad går alltid till samma modell)
# ------------------------------------------------------------
np.random.seed(42)
n_test = len(X_test_scaled)

# Skapa routing: True = v1, False = v2
routing = np.array([hash(i) % 10 < 7 for i in range(n_test)])

# Gör förutsägelser per modell baserat på routing
ab_pred = np.where(routing,
                   rf_pred,    # v1 för 70%
                   rf_pred_v2) # v2 för 30%

# Räkna korrekthet per version
v1_correct = (rf_pred[routing]    == y_test.values[routing]).mean()
v2_correct = (rf_pred_v2[~routing] == y_test.values[~routing]).mean()

print("=" * 50)
print("A/B-TEST RESULTAT")
print("=" * 50)
print(f"\nVersion A (v1) — 100 träd, djup 10:")
print(f"  F1-score:   {rf_f1:.3f}")
print(f"  Korrekthet: {v1_correct:.3f} ({routing.sum()} rader)")

print(f"\nVersion B (v2) — 200 träd, djup 15:")
print(f"  F1-score:   {rf_f1_v2:.3f}")
print(f"  Korrekthet: {v2_correct:.3f} ({(~routing).sum()} rader)")

print(f"\nRekommendation: ", end="")
if rf_f1 >= rf_f1_v2:
    print("Behåll Version A (v1) i produktion")
else:
    print("Uppgradera till Version B (v2)")

2026/06/11 08:23:36 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://dbc-40a283be-55c0.cloud.databricks.com/ml/experiments/1918190157109294/models/m-3dfb5bb26b884724b50666736d93fc63?o=7474659315390665
2026/06/11 08:23:39 INFO mlflow.models.model: Model logged without a signature. Signatures are required for Databricks UC model registry as they validate model inputs and denote the expected schema of model outputs. Please set `input_example` parameter when logging the model to auto infer the model signature. To manually set the signature, please visit https://www.mlflow.org/docs/3.8.1/ml/model/signatures.html for instructions on setting signature on models.


A/B-TEST RESULTAT

Version A (v1) — 100 träd, djup 10:
  F1-score:   0.995
  Korrekthet: 0.995 (17638 rader)

Version B (v2) — 200 träd, djup 15:
  F1-score:   0.997
  Korrekthet: 0.998 (7557 rader)

Rekommendation: Uppgradera till Version B (v2)


In [0]:
# ============================================================
# CELL 8: Skicka automatisk rapport till Slack
# ============================================================

import requests
import json

# Webhook-URL från Slack API — denna URL är hemlig!
# I produktion lagras den som miljövariabel, aldrig i koden
# OBS: Ta bort denna URL innan du pushar till GitHub!
SLACK_WEBHOOK_URL = "https://hooks.slack.com/services/XXX/YYY/ZZZ"  # Lägg i miljövariabel

# ------------------------------------------------------------
# Bygg rapporten som ett strukturerat Slack-meddelande
# Vi sammanfattar resultaten från alla modellkörningar
# ------------------------------------------------------------
rapport = f"""
*NSL-KDD Intrusion Detection — Modellrapport*
================================================

*Dataset:* 125 973 nätverksanslutningar
*Träningsdata:* 100 778 rader (80%)
*Testdata:* 25 195 rader (20%)

*Modellresultat:*
- Isolation Forest F1-score: {iso_f1:.3f}
- Random Forest v1 F1-score: {rf_f1:.3f}
- Random Forest v2 F1-score: {rf_f1_v2:.3f}

*A/B-test (70/30 trafikdelning):*
- Version A (v1) korrekthet: {v1_correct:.3f}
- Version B (v2) korrekthet: {v2_correct:.3f}
- Rekommendation: {"Behåll v1" if rf_f1 >= rf_f1_v2 else "Uppgradera till v2"}

*Attackfördelning i testdata:*
- Normal:  {(y_test == 0).sum()} rader
- Attack:  {(y_test == 1).sum()} rader

*MLflow Run ID (RF v1):* `{rf_run_id}`
*Status:* Klar
"""

# ------------------------------------------------------------
# Skicka rapporten till Slack via HTTP POST
# requests.post skickar ett HTTP-anrop till Slack-servern
# json.dumps konverterar Python-dict till JSON-format
# ------------------------------------------------------------
response = requests.post(
    SLACK_WEBHOOK_URL,
    data=json.dumps({"text": rapport}),
    headers={"Content-Type": "application/json"}
)

# Kontrollera att meddelandet skickades korrekt
# HTTP 200 = lyckades, annat = fel
if response.status_code == 200:
    print("Slack-rapport skickad!")
else:
    print(f"Fel: {response.status_code} — {response.text}")

Slack-rapport skickad!


In [0]:
# ============================================================
# CELL 9: Skapa permanent tabell med modellhistorik
# ============================================================
# Dashboarden i Databricks SQL behöver en tabell att läsa från
# Vi skapar en DataFrame med alla körningar och sparar den
# som en permanent Delta-tabell i Databricks

# Skapa en DataFrame med modellhistoriken
import pandas as pd
from datetime import datetime

historik = pd.DataFrame([
    {
        "run_name":      "Isolation_Forest",
        "model_type":    "IsolationForest",
        "n_estimators":  None,
        "max_depth":     None,
        "f1_score":      round(iso_f1, 4),
        "precision":     round(precision_score(y_test, iso_pred), 4),
        "recall":        round(recall_score(y_test, iso_pred), 4),
        "run_date":      datetime.now().strftime("%Y-%m-%d")
    },
    {
        "run_name":      "Random_Forest_v1",
        "model_type":    "RandomForest",
        "n_estimators":  100,
        "max_depth":     10,
        "f1_score":      round(rf_f1, 4),
        "precision":     round(precision_score(y_test, rf_pred), 4),
        "recall":        round(recall_score(y_test, rf_pred), 4),
        "run_date":      datetime.now().strftime("%Y-%m-%d")
    },
    {
        "run_name":      "Random_Forest_v2",
        "model_type":    "RandomForest",
        "n_estimators":  200,
        "max_depth":     15,
        "f1_score":      round(rf_f1_v2, 4),
        "precision":     round(precision_score(y_test, rf_pred_v2), 4),
        "recall":        round(recall_score(y_test, rf_pred_v2), 4),
        "run_date":      datetime.now().strftime("%Y-%m-%d")
    }
])

# Konvertera till Spark DataFrame och spara som Delta-tabell
# Delta är Databricks inbyggda tabellformat — stöder SQL-frågor
spark_df = spark.createDataFrame(historik)
spark_df.write.mode("overwrite").saveAsTable("model_performance_history")

print("Tabell skapad!")
print(historik.to_string(index=False))

Tabell skapad!
        run_name      model_type  n_estimators  max_depth  f1_score  precision  recall   run_date
Isolation_Forest IsolationForest           NaN        NaN    0.6424     0.6456  0.6392 2026-06-11
Random_Forest_v1    RandomForest         100.0       10.0    0.9954     0.9986  0.9923 2026-06-11
Random_Forest_v2    RandomForest         200.0       15.0    0.9972     0.9991  0.9952 2026-06-11
